In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 140)

WORKING_DIR = Path.cwd()
PROJECT_ROOT = (
    WORKING_DIR.parent
    if WORKING_DIR.name == "notebooks"
    else WORKING_DIR
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
SQL_DIR = PROJECT_ROOT / "sql"

for folder in (RAW_DIR, PROCESSED_DIR, NOTEBOOKS_DIR, SQL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

FILE_PATH = RAW_DIR / "Online_Retail.xlsx"
SHEET_NAME = "Online Retail"

assert FILE_PATH.is_file(), f"File not found: {FILE_PATH}"

with pd.ExcelFile(FILE_PATH, engine="openpyxl") as workbook:
    assert SHEET_NAME in workbook.sheet_names, workbook.sheet_names
    df_raw = pd.read_excel(workbook, sheet_name=SHEET_NAME)

EXPECTED_COLUMNS = [
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country"
]

assert df_raw.columns.tolist() == EXPECTED_COLUMNS, "Unexpected columns."
assert not df_raw.empty, "The dataset is empty."

df = df_raw.copy(deep=True)
assert df.equals(df_raw), "The working copy differs from the raw data."

print(f"File: {FILE_PATH}")
print(f"Sheet: {SHEET_NAME}")
print(f"Shape: {df_raw.shape}")

display(df_raw.head(5))
display(df_raw.dtypes.rename("dtype").to_frame())

File: d:\project  whith Alex\E-Commerce Sales & Customer Analytics\data\raw\Online_Retail.xlsx
Sheet: Online Retail
Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


,dtype
InvoiceNo,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[ns]
UnitPrice,float64
CustomerID,float64
Country,object


In [2]:
print("Working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Expected file:", FILE_PATH)
print("Excel files found:")

excel_files = sorted(
    path
    for path in PROJECT_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in {".xlsx", ".xls"}
)

for path in excel_files:
    print(path.relative_to(PROJECT_ROOT))

if not excel_files:
    print("No Excel files found inside the project folder.")

Working directory: d:\project  whith Alex\E-Commerce Sales & Customer Analytics
Project root: d:\project  whith Alex\E-Commerce Sales & Customer Analytics
Expected file: d:\project  whith Alex\E-Commerce Sales & Customer Analytics\data\raw\Online_Retail.xlsx
Excel files found:
Online Retail.xlsx


In [3]:
import shutil

source_path = PROJECT_ROOT / "Online Retail.xlsx"
target_path = RAW_DIR / "Online_Retail.xlsx"

assert source_path.is_file(), f"Source file not found: {source_path}"
assert not target_path.exists(), f"Target already exists: {target_path}"

shutil.copy2(source_path, target_path)

assert target_path.is_file()
assert source_path.stat().st_size == target_path.stat().st_size

print(f"Copied to: {target_path}")

Copied to: d:\project  whith Alex\E-Commerce Sales & Customer Analytics\data\raw\Online_Retail.xlsx


In [5]:
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")

quality_summary = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_count": df_raw.isna().sum(),
    "missing_pct": df_raw.isna().mean().mul(100).round(2)
})

display(quality_summary)

print(f"Exact duplicate rows beyond first: {df_raw.duplicated().sum():,}")
print(f"First invoice date: {df_raw['InvoiceDate'].min()}")
print(f"Last invoice date: {df_raw['InvoiceDate'].max()}")

Rows: 541,909
Columns: 8


,dtype,missing_count,missing_pct
InvoiceNo,object,0,0.00
StockCode,object,0,0.00
Description,object,1454,0.27
Quantity,int64,0,0.00
InvoiceDate,datetime64[ns],0,0.00
UnitPrice,float64,0,0.00
CustomerID,float64,135080,24.93
Country,object,0,0.00


Exact duplicate rows beyond first: 5,268
First invoice date: 2010-12-01 08:26:00
Last invoice date: 2011-12-09 12:50:00


In [6]:
invoice_text = df_raw["InvoiceNo"].astype("string").str.strip()

checks = pd.DataFrame({
    "C_invoice": invoice_text.str.upper().str.startswith("C"),
    "A_invoice": invoice_text.str.upper().str.startswith("A"),
    "Negative_quantity": df_raw["Quantity"].lt(0),
    "Negative_quantity_without_C": (
        df_raw["Quantity"].lt(0)
        & ~invoice_text.str.upper().str.startswith("C")
    ),
    "C_invoice_nonnegative_quantity": (
        invoice_text.str.upper().str.startswith("C")
        & df_raw["Quantity"].ge(0)
    ),
    "Zero_quantity": df_raw["Quantity"].eq(0),
    "Zero_price": df_raw["UnitPrice"].eq(0),
    "Negative_price": df_raw["UnitPrice"].lt(0)
})

display(pd.DataFrame({
    "row_count": checks.sum(),
    "row_pct": checks.mean().mul(100).round(2)
}))

for check_name in [
    "Negative_quantity_without_C",
    "A_invoice",
    "Zero_price",
    "Negative_price"
]:
    print(f"\n{check_name}:")
    display(df_raw.loc[checks[check_name]].head(5))

,row_count,row_pct
C_invoice,9288,1.71
A_invoice,3,0.0
Negative_quantity,10624,1.96
Negative_quantity_without_C,1336,0.25
C_invoice_nonnegative_quantity,0,0.0
Zero_quantity,0,0.0
Zero_price,2515,0.46
Negative_price,2,0.0



Negative_quantity_without_C:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom



A_invoice:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom



Zero_price:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom



Negative_price:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [7]:
review = df_raw.assign(
    InvoicePrefix=invoice_text.str.upper().str[0],
    QuantitySign=np.where(df_raw["Quantity"].lt(0), "Negative", "Positive"),
    PriceSign=np.select(
        [df_raw["UnitPrice"].lt(0), df_raw["UnitPrice"].eq(0)],
        ["Negative", "Zero"],
        default="Positive"
    ),
    MissingCustomer=df_raw["CustomerID"].isna(),
    MissingDescription=df_raw["Description"].isna()
)

for label, mask in {
    "Negative quantity without C": checks["Negative_quantity_without_C"],
    "Zero price": checks["Zero_price"]
}.items():
    print(f"\n{label}")

    summary = (
        review.loc[mask]
        .groupby(
            ["QuantitySign", "PriceSign", "MissingCustomer", "MissingDescription"],
            dropna=False
        )
        .size()
        .rename("row_count")
        .reset_index()
        .sort_values("row_count", ascending=False)
    )

    assert summary["row_count"].sum() == mask.sum()
    display(summary)

print("\nMost frequent descriptions: negative quantity without C")
display(
    df_raw.loc[checks["Negative_quantity_without_C"], "Description"]
    .value_counts(dropna=False)
    .head(15)
    .rename("row_count")
    .to_frame()
)


Negative quantity without C


,QuantitySign,PriceSign,MissingCustomer,MissingDescription,row_count
1,Negative,Zero,True,True,862
0,Negative,Zero,True,False,474



Zero price


,QuantitySign,PriceSign,MissingCustomer,MissingDescription,row_count
1,Negative,Zero,True,True,862
4,Positive,Zero,True,True,592
3,Positive,Zero,True,False,547
0,Negative,Zero,True,False,474
2,Positive,Zero,False,False,40



Most frequent descriptions: negative quantity without C


,row_count
Description,
NaN,862
check,120
damages,45
damaged,42
?,41
sold as set on dotcom,20
Damaged,14
thrown away,9
"Unsaleable, destroyed.",9


In [8]:
zero_price_known_customer = df_raw.loc[
    df_raw["UnitPrice"].eq(0)
    & df_raw["Quantity"].gt(0)
    & df_raw["CustomerID"].notna()
].copy()

print(f"Rows: {len(zero_price_known_customer):,}")
print(
    "Distinct invoices:",
    zero_price_known_customer["InvoiceNo"].nunique()
)

with pd.option_context("display.max_rows", None, "display.max_colwidth", 100):
    display(
        zero_price_known_customer.sort_values(
            ["InvoiceDate", "InvoiceNo"]
        )
    )

Rows: 40
Distinct invoices: 34


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.0,12647.0,Germany
33576,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,2010-12-16 14:36:00,0.0,16560.0,United Kingdom
40089,539722,22423,REGENCY CAKESTAND 3 TIER,10,2010-12-21 13:45:00,0.0,14911.0,EIRE
47068,540372,22090,PAPER BUNTING RETROSPOT,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom
47070,540372,22553,PLASTERS IN TIN SKULLS,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom
56674,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,2011-01-13 15:10:00,0.0,15107.0,United Kingdom
86789,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,2011-02-10 13:08:00,0.0,17560.0,United Kingdom
130188,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,2011-03-23 10:25:00,0.0,13239.0,United Kingdom
139453,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,2011-03-30 12:45:00,0.0,13113.0,United Kingdom
145208,548871,22162,HEART GARLAND RUSTIC PADDED,2,2011-04-04 14:42:00,0.0,14410.0,United Kingdom


In [9]:
stock_text = df_raw["StockCode"].astype("string").str.strip()

unusual_stock_mask = ~stock_text.str.fullmatch(
    r"\d{5}[A-Za-z]*", na=False
)

stock_review = (
    df_raw.loc[unusual_stock_mask]
    .assign(StockCodeText=stock_text.loc[unusual_stock_mask])
    .groupby("StockCodeText", dropna=False)
    .agg(
        row_count=("StockCode", "size"),
        descriptions=(
            "Description",
            lambda values: " | ".join(
                sorted(values.dropna().astype(str).str.strip().unique())
            )
        ),
        min_price=("UnitPrice", "min"),
        max_price=("UnitPrice", "max")
    )
    .sort_values("row_count", ascending=False)
)

assert stock_review["row_count"].sum() == unusual_stock_mask.sum()

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(stock_review)

,row_count,descriptions,min_price,max_price
StockCodeText,,,,
POST,1256,POSTAGE,0.000,8142.750
DOT,710,DOTCOM POSTAGE,0.000,4505.170
M,571,Manual,0.000,38970.000
C2,144,CARRIAGE,0.000,150.000
D,77,Discount,0.010,1867.860
S,63,SAMPLES,2.800,570.000
BANK CHARGES,37,Bank Charges,0.001,1050.150
AMAZONFEE,34,AMAZON FEE,1.000,17836.460
CRUK,16,CRUK Commission,1.600,1100.440


In [10]:
duplicate_extra = df_raw.duplicated(keep="first")
duplicate_all = df_raw.duplicated(keep=False)
raw_line_value = df_raw["Quantity"] * df_raw["UnitPrice"]

duplicate_impact = pd.Series({
    "extra_duplicate_rows": int(duplicate_extra.sum()),
    "rows_in_duplicate_groups": int(duplicate_all.sum()),
    "invoices_with_extra_duplicates": (
        df_raw.loc[duplicate_extra, "InvoiceNo"].nunique()
    ),
    "extra_duplicates_signed_value": (
        raw_line_value.loc[duplicate_extra].sum()
    ),
    "extra_duplicates_absolute_value": (
        raw_line_value.loc[duplicate_extra].abs().sum()
    )
})

display(duplicate_impact.to_frame("value"))

duplicate_examples = (
    df_raw.loc[duplicate_all]
    .assign(InvoiceNoText=invoice_text.loc[duplicate_all])
    .sort_values(["InvoiceNoText", "StockCode", "InvoiceDate"])
    .drop(columns="InvoiceNoText")
    .head(20)
)

display(duplicate_examples)

,value
extra_duplicate_rows,5268.00
rows_in_duplicate_groups,10147.00
invoices_with_extra_duplicates,1933.00
extra_duplicates_signed_value,21740.98
extra_duplicates_absolute_value,27406.50


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920.0,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom


In [11]:
rows_before = len(df_raw)

df = df_raw.copy(deep=True)

df["IsExactDuplicate"] = df_raw.duplicated(keep="first")
df["SourceRowID"] = np.arange(2, rows_before + 2)

assert len(df) == rows_before
assert df["SourceRowID"].is_unique
assert df[EXPECTED_COLUMNS].equals(df_raw)
assert df["IsExactDuplicate"].sum() == df_raw.duplicated().sum()
assert not df.loc[
    ~df["IsExactDuplicate"], EXPECTED_COLUMNS
].duplicated().any()

duplicate_summary = pd.Series({
    "Rows before": rows_before,
    "Rows after": len(df),
    "Flagged extra duplicates": int(df["IsExactDuplicate"].sum()),
    "Rows excluding extra duplicates": int((~df["IsExactDuplicate"]).sum())
}, name="row_count")

display(duplicate_summary.to_frame())

,row_count
Rows before,541909
Rows after,541909
Flagged extra duplicates,5268
Rows excluding extra duplicates,536641


In [12]:
rows_before = len(df)
df_before_classification = df.copy(deep=True)

stock_key = df["StockCode"].astype("string").str.strip().str.upper()
invoice_key = df["InvoiceNo"].astype("string").str.strip().str.upper()

category_map = {
    "POST": "Shipping",
    "DOT": "Shipping",
    "C2": "Shipping",
    "BANK CHARGES": "Fees",
    "AMAZONFEE": "Fees",
    "CRUK": "Fees",
    "D": "Discount",
    "M": "Manual",
    "S": "Samples",
    "B": "DebtAdjustment"
}

reviewed_dcgs_products = {
    "DCGSSGIRL", "DCGSSBOY", "DCGS0003",
    "DCGS0069", "DCGS0070", "DCGS0076", "DCGS0004"
}

product_candidate = (
    stock_key.str.fullmatch(r"\d{5}[A-Z]*", na=False)
    | stock_key.isin(reviewed_dcgs_products)
    | stock_key.eq("PADS")
)

df["LineCategory"] = stock_key.map(category_map).fillna("Unclassified")

df.loc[product_candidate, "LineCategory"] = "Product"
df.loc[stock_key.str.startswith("GIFT_", na=False), "LineCategory"] = "GiftVoucher"
df.loc[invoice_key.str.startswith("A", na=False), "LineCategory"] = "DebtAdjustment"

assert len(df) == rows_before
assert df[df_before_classification.columns].equals(df_before_classification)
assert df["LineCategory"].notna().all()

category_summary = (
    df.groupby("LineCategory")
    .agg(
        total_rows=("SourceRowID", "size"),
        extra_duplicates=("IsExactDuplicate", "sum")
    )
)

category_summary["rows_without_duplicates"] = (
    category_summary["total_rows"] - category_summary["extra_duplicates"]
)

assert category_summary["total_rows"].sum() == rows_before

print(f"Rows before: {rows_before:,} | Rows after: {len(df):,}")
display(category_summary.sort_values("total_rows", ascending=False))

Rows before: 541,909 | Rows after: 541,909


,total_rows,extra_duplicates,rows_without_duplicates
LineCategory,,,
Product,538954,5262,533692
Shipping,2110,0,2110
Manual,572,5,567
Fees,87,0,87
Discount,77,0,77
Samples,63,1,62
GiftVoucher,34,0,34
Unclassified,9,0,9
DebtAdjustment,3,0,3


In [13]:
rows_before = len(df)
df_before_flags = df.copy(deep=True)

invoice_key = df["InvoiceNo"].astype("string").str.strip().str.upper()

df["IsCancellation"] = invoice_key.str.startswith("C", na=False)
df["IsDebtAdjustment"] = df["LineCategory"].eq("DebtAdjustment")
df["IsZeroPrice"] = df["UnitPrice"].eq(0)
df["IsNegativePrice"] = df["UnitPrice"].lt(0)
df["HasCustomerID"] = df["CustomerID"].notna()
df["IsPotentialStockAdjustment"] = (
    df["Quantity"].lt(0)
    & ~df["IsCancellation"]
    & df["IsZeroPrice"]
)

flag_columns = [
    "IsCancellation",
    "IsDebtAdjustment",
    "IsZeroPrice",
    "IsNegativePrice",
    "HasCustomerID",
    "IsPotentialStockAdjustment"
]

assert len(df) == rows_before
assert df[df_before_flags.columns].equals(df_before_flags)
assert not df[flag_columns].isna().any().any()
assert df.loc[df["IsCancellation"], "Quantity"].lt(0).all()
assert df.loc[df["IsNegativePrice"], "IsDebtAdjustment"].all()

flag_summary = pd.DataFrame({
    "all_rows": df[flag_columns].sum(),
    "rows_without_duplicates": (
        df.loc[~df["IsExactDuplicate"], flag_columns].sum()
    )
})

print(f"Rows before: {rows_before:,} | Rows after: {len(df):,}")
display(flag_summary)

Rows before: 541,909 | Rows after: 541,909


,all_rows,rows_without_duplicates
IsCancellation,9288,9251
IsDebtAdjustment,3,3
IsZeroPrice,2515,2510
IsNegativePrice,2,2
HasCustomerID,406829,401604
IsPotentialStockAdjustment,1336,1336


In [14]:
rows_before = len(df)
df_before_eligibility = df.copy(deep=True)

product_base = (
    ~df["IsExactDuplicate"]
    & df["LineCategory"].eq("Product")
    & ~df["IsDebtAdjustment"]
    & df["UnitPrice"].gt(0)
)

df["IsProductSale"] = (
    product_base
    & df["Quantity"].gt(0)
    & ~df["IsCancellation"]
)

df["IsProductCancellation"] = (
    product_base
    & df["Quantity"].lt(0)
    & df["IsCancellation"]
)

df["IsNetProductRevenueEligible"] = (
    df["IsProductSale"] | df["IsProductCancellation"]
)

df["IsCustomerPurchaseEligible"] = (
    df["IsProductSale"] & df["HasCustomerID"]
)

assert len(df) == rows_before
assert df[df_before_eligibility.columns].equals(df_before_eligibility)
assert not (df["IsProductSale"] & df["IsProductCancellation"]).any()
assert not df.loc[
    df["IsNetProductRevenueEligible"], "IsExactDuplicate"
].any()
assert df.loc[
    df["IsCustomerPurchaseEligible"], "CustomerID"
].notna().all()

eligibility_columns = [
    "IsProductSale",
    "IsProductCancellation",
    "IsNetProductRevenueEligible",
    "IsCustomerPurchaseEligible"
]

eligibility_summary = pd.DataFrame({
    "eligible_rows": df[eligibility_columns].sum(),
    "rows_without_customer": (
        df.loc[~df["HasCustomerID"], eligibility_columns].sum()
    )
})

print(f"Rows before: {rows_before:,} | Rows after: {len(df):,}")
display(eligibility_summary)

Rows before: 541,909 | Rows after: 541,909


,eligible_rows,rows_without_customer
IsProductSale,522540,131387
IsProductCancellation,8668,162
IsNetProductRevenueEligible,531208,131549
IsCustomerPurchaseEligible,391153,0


In [15]:
rows_before = len(df)
df_before_features = df.copy(deep=True)

df["Revenue"] = df["Quantity"] * df["UnitPrice"]
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M").astype("string")
df["Weekday"] = df["InvoiceDate"].dt.dayofweek
df["Hour"] = df["InvoiceDate"].dt.hour

assert len(df) == rows_before
assert df[df_before_features.columns].equals(df_before_features)
assert np.isfinite(df["Revenue"]).all()
assert df["Month"].between(1, 12).all()
assert df["Weekday"].between(0, 6).all()
assert df["Hour"].between(0, 23).all()
assert df.loc[df["IsProductSale"], "Revenue"].gt(0).all()
assert df.loc[df["IsProductCancellation"], "Revenue"].lt(0).all()

gross_product_revenue = df.loc[df["IsProductSale"], "Revenue"].sum()
cancellation_value = -df.loc[df["IsProductCancellation"], "Revenue"].sum()
net_product_revenue = df.loc[
    df["IsNetProductRevenueEligible"], "Revenue"
].sum()

assert np.isclose(
    gross_product_revenue - cancellation_value,
    net_product_revenue,
    rtol=0,
    atol=0.000001
)

print(f"Rows before: {rows_before:,} | Rows after: {len(df):,}")

display(pd.Series({
    "Gross product revenue": gross_product_revenue,
    "Product cancellation value": cancellation_value,
    "Net product revenue": net_product_revenue
}, name="value").to_frame().style.format("{:,.3f}"))

display(
    df[[
        "InvoiceDate", "Quantity", "UnitPrice", "Revenue",
        "Year", "Month", "YearMonth", "Weekday", "Hour"
    ]].head()
)

Rows before: 541,909 | Rows after: 541,909


,value
Gross product revenue,"10,247,219.323"
Product cancellation value,"475,901.160"
Net product revenue,"9,771,318.163"


,InvoiceDate,Quantity,UnitPrice,Revenue,Year,Month,YearMonth,Weekday,Hour
0,2010-12-01 08:26:00,6,2.55,15.30,2010,12,2010-12,2,8
1,2010-12-01 08:26:00,6,3.39,20.34,2010,12,2010-12,2,8
2,2010-12-01 08:26:00,8,2.75,22.00,2010,12,2010-12,2,8
3,2010-12-01 08:26:00,6,3.39,20.34,2010,12,2010-12,2,8
4,2010-12-01 08:26:00,6,3.39,20.34,2010,12,2010-12,2,8


In [16]:
rows_before = len(df)
df_before_types = df.copy(deep=True)

known_customer_ids = df["CustomerID"].dropna()
assert known_customer_ids.mod(1).eq(0).all()

df["InvoiceNo"] = df["InvoiceNo"].astype("string")
df["StockCode"] = df["StockCode"].astype("string")
df["CustomerID"] = df["CustomerID"].astype("Int64").astype("string")

assert len(df) == rows_before

for column in ["InvoiceNo", "StockCode", "CustomerID"]:
    assert df[column].isna().equals(df_before_types[column].isna())

assert (
    df["CustomerID"].dropna().astype("int64")
    .eq(known_customer_ids.astype("int64"))
    .all()
)

unchanged_columns = df_before_types.columns.difference(
    ["InvoiceNo", "StockCode", "CustomerID"]
)

assert df[unchanged_columns].equals(df_before_types[unchanged_columns])

print(f"Rows before: {rows_before:,} | Rows after: {len(df):,}")

display(
    df[["InvoiceNo", "StockCode", "CustomerID"]]
    .dtypes.rename("dtype").to_frame()
)

display(df[["InvoiceNo", "StockCode", "CustomerID"]].head())

print(f"Missing CustomerID: {df['CustomerID'].isna().sum():,}")

Rows before: 541,909 | Rows after: 541,909


,dtype
InvoiceNo,string[python]
StockCode,string[python]
CustomerID,string[python]


,InvoiceNo,StockCode,CustomerID
0,536365,85123A,17850
1,536365,71053,17850
2,536365,84406B,17850
3,536365,84029G,17850
4,536365,84029E,17850


Missing CustomerID: 135,080


In [17]:
invoice_audit = (
    df.loc[~df["IsExactDuplicate"]]
    .groupby("InvoiceNo")
    .agg(
        customer_count=("CustomerID", "nunique"),
        country_count=("Country", "nunique"),
        timestamp_count=("InvoiceDate", "nunique"),
        has_missing_customer=("CustomerID", lambda values: values.isna().any())
    )
)

invoice_checks = pd.Series({
    "Invoices with multiple known customers": (
        invoice_audit["customer_count"].gt(1).sum()
    ),
    "Invoices with multiple countries": (
        invoice_audit["country_count"].gt(1).sum()
    ),
    "Invoices with multiple timestamps": (
        invoice_audit["timestamp_count"].gt(1).sum()
    ),
    "Invoices mixing known and missing customers": (
        invoice_audit["customer_count"].gt(0)
        & invoice_audit["has_missing_customer"]
    ).sum()
})

display(invoice_checks.to_frame("invoice_count"))

invoice_issues = invoice_audit.loc[
    invoice_audit["customer_count"].gt(1)
    | invoice_audit["country_count"].gt(1)
    | invoice_audit["timestamp_count"].gt(1)
    | (
        invoice_audit["customer_count"].gt(0)
        & invoice_audit["has_missing_customer"]
    )
]

display(invoice_issues.head(10))

product_description_counts = (
    df.loc[df["IsNetProductRevenueEligible"]]
    .assign(
        DescriptionChecked=lambda data: (
            data["Description"].astype("string").str.strip().str.upper()
        )
    )
    .groupby("StockCode")["DescriptionChecked"]
    .nunique()
    .sort_values(ascending=False)
)

print(
    "Product codes with multiple descriptions:",
    product_description_counts.gt(1).sum()
)

display(
    product_description_counts.loc[product_description_counts.gt(1)]
    .head(10)
    .rename("description_count")
    .to_frame()
)

,invoice_count
Invoices with multiple known customers,0
Invoices with multiple countries,0
Invoices with multiple timestamps,43
Invoices mixing known and missing customers,0


,customer_count,country_count,timestamp_count,has_missing_customer
InvoiceNo,,,,
536591,1,1,2,False
540185,1,1,2,False
541596,1,1,2,False
541631,1,1,2,False
541809,0,1,2,True
541816,1,1,2,False
541849,1,1,2,False
542217,1,1,2,False
542806,1,1,2,False


Product codes with multiple descriptions: 217


,description_count
StockCode,
23196,4
23236,4
17107D,3
23535,3
23231,3
23366,3
23370,3
23209,3
23203,3


In [18]:
multi_time_ids = invoice_audit.index[
    invoice_audit["timestamp_count"].gt(1)
]

invoice_time_review = (
    df.loc[
        ~df["IsExactDuplicate"]
        & df["InvoiceNo"].isin(multi_time_ids)
    ]
    .groupby("InvoiceNo")
    .agg(
        first_time=("InvoiceDate", "min"),
        last_time=("InvoiceDate", "max"),
        row_count=("SourceRowID", "size")
    )
)

invoice_time_review["gap_minutes"] = (
    invoice_time_review["last_time"]
    - invoice_time_review["first_time"]
).dt.total_seconds().div(60)

invoice_time_review["crosses_day"] = (
    invoice_time_review["first_time"].dt.normalize()
    != invoice_time_review["last_time"].dt.normalize()
)

invoice_time_review["crosses_month"] = (
    invoice_time_review["first_time"].dt.to_period("M")
    != invoice_time_review["last_time"].dt.to_period("M")
)

print("Invoices crossing days:", invoice_time_review["crosses_day"].sum())
print("Invoices crossing months:", invoice_time_review["crosses_month"].sum())

display(
    invoice_time_review.sort_values("gap_minutes", ascending=False).head(10)
)

sample_codes = product_description_counts.loc[
    product_description_counts.gt(1)
].head(5).index

description_review = (
    df.loc[
        df["IsNetProductRevenueEligible"]
        & df["StockCode"].isin(sample_codes)
    ]
    .assign(
        DescriptionChecked=lambda data: (
            data["Description"].astype("string").str.strip().str.upper()
        )
    )
    .groupby(["StockCode", "DescriptionChecked"], dropna=False)
    .agg(
        row_count=("SourceRowID", "size"),
        first_seen=("InvoiceDate", "min"),
        last_seen=("InvoiceDate", "max")
    )
    .reset_index()
)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(description_review)

Invoices crossing days: 0
Invoices crossing months: 0


,first_time,last_time,row_count,gap_minutes,crosses_day,crosses_month
InvoiceNo,,,,,,
536591,2010-12-01 16:57:00,2010-12-01 16:58:00,40,1.0,False,False
554116,2011-05-22 15:11:00,2011-05-22 15:12:00,13,1.0,False,False
550320,2011-04-17 12:37:00,2011-04-17 12:38:00,83,1.0,False,False
550333,2011-04-17 14:05:00,2011-04-17 14:06:00,22,1.0,False,False
550641,2011-04-19 15:53:00,2011-04-19 15:54:00,9,1.0,False,False
550645,2011-04-19 16:30:00,2011-04-19 16:31:00,7,1.0,False,False
552000,2011-05-05 15:55:00,2011-05-05 15:56:00,143,1.0,False,False
553199,2011-05-15 15:13:00,2011-05-15 15:14:00,58,1.0,False,False
553375,2011-05-16 14:52:00,2011-05-16 14:53:00,39,1.0,False,False


,StockCode,DescriptionChecked,row_count,first_seen,last_seen
0,17107D,FLOWER FAIRY 5 DRAWER LINERS,20,2011-08-28 13:32:00,2011-12-05 15:51:00
1,17107D,FLOWER FAIRY 5 SUMMER DRAW LINERS,1,2011-09-19 15:00:00,2011-09-19 15:00:00
2,17107D,"FLOWER FAIRY,5 SUMMER B'DRAW LINERS",25,2010-12-03 12:24:00,2011-06-19 14:42:00
3,23196,LEAVES MAGNETIC SHOPPING LIST,2,2011-07-01 14:53:00,2011-07-14 12:22:00
4,23196,RETO LEAVES MAGNETIC SHOPPING LIST,3,2011-06-24 09:22:00,2011-09-22 14:26:00
5,23196,RETRO LEAVES MAGNETIC NOTEPAD,21,2011-06-23 11:51:00,2011-07-20 10:51:00
6,23196,VINTAGE LEAF MAGNETIC NOTEPAD,281,2011-07-06 13:56:00,2011-12-09 10:26:00
7,23231,WRAP VINTAGE DOILEY,2,2011-08-05 11:00:00,2011-08-09 13:51:00
8,23231,WRAP VINTAGE DOILY,94,2011-08-09 14:18:00,2011-12-08 14:14:00
9,23231,WRAP DOILEY DESIGN,164,2011-01-12 15:06:00,2011-08-18 08:49:00


In [19]:
rows_before = len(df)
df_before_labels = df.copy(deep=True)

df["InvoiceTimestamp"] = (
    df.groupby("InvoiceNo")["InvoiceDate"].transform("min")
)

description_candidates = (
    df.loc[
        df["IsNetProductRevenueEligible"],
        ["StockCode", "Description", "InvoiceDate"]
    ]
    .assign(
        DisplayName=lambda data: (
            data["Description"]
            .astype("string")
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .str.upper()
        )
    )
    .dropna(subset=["DisplayName"])
)

description_candidates = description_candidates.loc[
    description_candidates["DisplayName"].ne("")
]

product_names = (
    description_candidates
    .groupby(["StockCode", "DisplayName"])
    .agg(
        occurrence_count=("InvoiceDate", "size"),
        last_seen=("InvoiceDate", "max")
    )
    .reset_index()
    .sort_values(
        ["StockCode", "occurrence_count", "last_seen", "DisplayName"],
        ascending=[True, False, False, True]
    )
    .drop_duplicates("StockCode")
    .set_index("StockCode")["DisplayName"]
)

df["ProductDisplayName"] = df["StockCode"].map(product_names).astype("string")
df.loc[df["LineCategory"].ne("Product"), "ProductDisplayName"] = pd.NA

assert len(df) == rows_before
assert df[df_before_labels.columns].equals(df_before_labels)
assert df["InvoiceTimestamp"].notna().all()
assert df.groupby("InvoiceNo")["InvoiceTimestamp"].nunique().eq(1).all()
assert df["InvoiceTimestamp"].le(df["InvoiceDate"]).all()
assert product_names.index.is_unique
assert df.loc[
    df["IsNetProductRevenueEligible"], "ProductDisplayName"
].notna().all()

print(f"Rows before: {rows_before:,} | Rows after: {len(df):,}")
print(f"Product display names: {len(product_names):,}")

display(
    product_names.reindex(sample_codes)
    .rename("ProductDisplayName")
    .to_frame()
)

Rows before: 541,909 | Rows after: 541,909
Product display names: 3,922


,ProductDisplayName
StockCode,
23196,VINTAGE LEAF MAGNETIC NOTEPAD
23236,STORAGE TIN VINTAGE DOILY
17107D,"FLOWER FAIRY,5 SUMMER B'DRAW LINERS"
23535,WALL ART BICYCLE SAFETY
23231,WRAP DOILEY DESIGN


In [20]:
eligible_review = df.loc[df["IsNetProductRevenueEligible"]].copy()

display(
    eligible_review[["Quantity", "UnitPrice", "Revenue"]]
    .describe(percentiles=[0.01, 0.5, 0.95, 0.99, 0.999])
    .round(3)
)

largest_line_indices = (
    eligible_review["Revenue"].abs().nlargest(10).index
)

display(
    eligible_review.loc[
        largest_line_indices,
        [
            "SourceRowID", "InvoiceNo", "StockCode",
            "ProductDisplayName", "Quantity", "UnitPrice",
            "Revenue", "InvoiceDate", "CustomerID",
            "Country", "IsCancellation"
        ]
    ]
)

print(
    "Eligible product rows priced below 0.01:",
    eligible_review["UnitPrice"].lt(0.01).sum()
)

,Quantity,UnitPrice,Revenue
count,531208.000,531208.000,531208.000
mean,9.961,3.293,18.395
std,217.007,4.513,369.755
min,-80995.000,0.001,-168469.600
1%,-1.000,0.290,-5.100
50%,3.000,2.080,9.900
95%,30.000,9.950,58.000
99%,100.000,16.630,179.000
99.9%,432.000,34.000,748.440
max,80995.000,649.500,168469.600


,SourceRowID,InvoiceNo,StockCode,ProductDisplayName,Quantity,UnitPrice,Revenue,InvoiceDate,CustomerID,Country,IsCancellation
540421,540423,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,2011-12-09 09:15:00,16446,United Kingdom,False
540422,540424,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,-168469.60,2011-12-09 09:27:00,16446,United Kingdom,True
61619,61621,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,2011-01-18 10:01:00,12346,United Kingdom,False
61624,61626,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,-77183.60,2011-01-18 10:17:00,12346,United Kingdom,True
222680,222682,556444,22502,PICNIC BASKET WICKER SMALL,60,649.50,38970.00,2011-06-10 15:28:00,15098,United Kingdom,False
348325,348327,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,5.06,7144.72,2011-09-20 11:05:00,17450,United Kingdom,False
52711,52713,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,6539.40,2011-01-11 12:55:00,15749,United Kingdom,False
160145,160147,C550456,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,-3114,2.10,-6539.40,2011-04-18 13:08:00,15749,United Kingdom,True
160546,160548,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,6539.40,2011-04-18 13:20:00,15749,United Kingdom,False
421601,421603,573003,23084,RABBIT NIGHT LIGHT,2400,2.08,4992.00,2011-10-27 12:11:00,14646,Netherlands,False


Eligible product rows priced below 0.01: 3


In [21]:
rows_before = len(df)
df_before_outlier_flags = df.copy(deep=True)

review_columns = ["Quantity", "UnitPrice", "Revenue"]

review_thresholds = (
    df.loc[df["IsProductSale"], review_columns]
    .quantile(0.999)
)

df["IsExtremeProductLine"] = (
    df["IsNetProductRevenueEligible"]
    & df[review_columns].abs().gt(review_thresholds).any(axis=1)
)

df["IsSubPennyProductPrice"] = (
    df["IsNetProductRevenueEligible"]
    & df["UnitPrice"].gt(0)
    & df["UnitPrice"].lt(0.01)
)

assert len(df) == rows_before
assert df[df_before_outlier_flags.columns].equals(df_before_outlier_flags)

print(f"Rows before: {rows_before:,} | Rows after: {len(df):,}")
display(review_thresholds.rename("review_threshold").to_frame())

display(
    df[["IsExtremeProductLine", "IsSubPennyProductPrice"]]
    .sum()
    .rename("flagged_rows")
    .to_frame()
)

display(
    df.loc[
        df["IsSubPennyProductPrice"],
        [
            "InvoiceNo", "StockCode", "Description",
            "Quantity", "UnitPrice", "Revenue", "CustomerID"
        ]
    ]
)

Rows before: 541,909 | Rows after: 541,909


,review_threshold
Quantity,456.00
UnitPrice,33.29
Revenue,750.00


,flagged_rows
IsExtremeProductLine,1437
IsSubPennyProductPrice,3


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,Revenue,CustomerID
157195,550193,PADS,PADS TO MATCH ALL CUSHIONS,1,0.001,0.001,13952
279045,561226,PADS,PADS TO MATCH ALL CUSHIONS,1,0.001,0.001,15618
359871,568200,PADS,PADS TO MATCH ALL CUSHIONS,1,0.001,0.001,16198


In [22]:
preserved_columns = [
    "Description", "Quantity", "InvoiceDate", "UnitPrice", "Country"
]

quality_checks = pd.Series({
    "All original rows retained": len(df) == len(df_raw),
    "Source row order preserved": np.array_equal(
        df["SourceRowID"].to_numpy(),
        np.arange(2, len(df_raw) + 2)
    ),
    "Original values preserved": df[preserved_columns].equals(
        df_raw[preserved_columns]
    ),
    "Invoice identifiers preserved": df["InvoiceNo"].equals(
        df_raw["InvoiceNo"].astype("string")
    ),
    "Product identifiers preserved": df["StockCode"].equals(
        df_raw["StockCode"].astype("string")
    ),
    "Customer identifiers preserved": df["CustomerID"].equals(
        df_raw["CustomerID"].astype("Int64").astype("string")
    ),
    "Duplicate flags match raw data": df["IsExactDuplicate"].equals(
        df_raw.duplicated(keep="first")
    ),
    "Revenue matches quantity times price": np.allclose(
        df["Revenue"],
        df["Quantity"] * df["UnitPrice"],
        rtol=0,
        atol=0.000001
    ),
    "Net scope matches sales and cancellations": (
        df["IsNetProductRevenueEligible"].equals(
            df["IsProductSale"] | df["IsProductCancellation"]
        )
    ),
    "Customer scope matches identified purchases": (
        df["IsCustomerPurchaseEligible"].equals(
            df["IsProductSale"] & df["CustomerID"].notna()
        )
    ),
    "Net product revenue reconciles": np.isclose(
        df.loc[df["IsProductSale"], "Revenue"].sum()
        + df.loc[df["IsProductCancellation"], "Revenue"].sum(),
        df.loc[df["IsNetProductRevenueEligible"], "Revenue"].sum(),
        rtol=0,
        atol=0.000001
    )
}, name="passed")

display(quality_checks.to_frame())

assert quality_checks.all(), (
    f"Failed checks: {quality_checks.index[~quality_checks].tolist()}"
)

print(f"Validation passed | Rows: {len(df):,} | Columns: {df.shape[1]}")

,passed
All original rows retained,True
Source row order preserved,True
Original values preserved,True
Invoice identifiers preserved,True
Product identifiers preserved,True
Customer identifiers preserved,True
Duplicate flags match raw data,True
Revenue matches quantity times price,True
Net scope matches sales and cancellations,True
Customer scope matches identified purchases,True


Validation passed | Rows: 541,909 | Columns: 31


In [25]:
rows_before = len(df)
df_before_description_fix = df.copy(deep=True)

df["Description"] = df["Description"].astype("string")

assert len(df) == rows_before
assert df["Description"].isna().equals(df_raw["Description"].isna())
assert df["Description"].equals(df_raw["Description"].astype("string"))

unchanged_columns = df_before_description_fix.columns.difference(
    ["Description"]
)

assert df[unchanged_columns].equals(
    df_before_description_fix[unchanged_columns]
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_path = PROCESSED_DIR / "online_retail_cleaned.parquet"

df.to_parquet(output_path, index=False, engine="pyarrow")
saved_df = pd.read_parquet(output_path, engine="pyarrow")

pd.testing.assert_frame_equal(
    df.reset_index(drop=True),
    saved_df.reset_index(drop=True),
    check_dtype=True,
    check_exact=True
)

print(f"Rows before: {rows_before:,} | Rows after: {len(df):,}")
print(f"Description dtype: {df['Description'].dtype}")
print(f"Missing descriptions: {df['Description'].isna().sum():,}")
print(f"Saved: {output_path}")
print(f"Shape: {saved_df.shape}")
print(f"File size: {output_path.stat().st_size / (1024 ** 2):.2f} MB")
print("Export verification passed.")

Rows before: 541,909 | Rows after: 541,909
Description dtype: string
Missing descriptions: 1,454
Saved: d:\project  whith Alex\E-Commerce Sales & Customer Analytics\data\processed\online_retail_cleaned.parquet
Shape: (541909, 31)
File size: 7.98 MB
Export verification passed.


In [26]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

try:
    import pyarrow
    parquet_engine = "pyarrow"
except ImportError:
    try:
        import fastparquet
        parquet_engine = "fastparquet"
    except ImportError:
        parquet_engine = None

if parquet_engine is not None:
    output_path = PROCESSED_DIR / "online_retail_cleaned.parquet"
    df.to_parquet(output_path, index=False, engine=parquet_engine)
    saved_df = pd.read_parquet(output_path, engine=parquet_engine)
else:
    output_path = PROCESSED_DIR / "online_retail_cleaned.csv"

    date_columns = df.select_dtypes(include=["datetime"]).columns.tolist()
    csv_dtypes = {
        column: str(dtype)
        for column, dtype in df.dtypes.items()
        if column not in date_columns
    }

    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    saved_df = pd.read_csv(
        output_path,
        dtype=csv_dtypes,
        parse_dates=date_columns,
        encoding="utf-8-sig",
        keep_default_na=False,
        na_values=[""],
        float_precision="round_trip"
    )

pd.testing.assert_frame_equal(
    df.reset_index(drop=True),
    saved_df.reset_index(drop=True),
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=0.000001
)

print(f"Saved: {output_path}")
print(f"Shape: {saved_df.shape}")
print(f"File size: {output_path.stat().st_size / (1024 ** 2):.2f} MB")
print("Export verification passed.")

Saved: d:\project  whith Alex\E-Commerce Sales & Customer Analytics\data\processed\online_retail_cleaned.parquet
Shape: (541909, 31)
File size: 7.98 MB
Export verification passed.


In [24]:
description_type_counts = (
    df["Description"]
    .map(lambda value: type(value).__name__)
    .value_counts()
    .rename("row_count")
    .to_frame()
)

non_text_description = (
    df["Description"].notna()
    & ~df["Description"].map(lambda value: isinstance(value, str))
)

display(description_type_counts)

print(f"Non-text descriptions: {non_text_description.sum():,}")

display(
    df.loc[
        non_text_description,
        [
            "SourceRowID", "InvoiceNo", "StockCode", "Description",
            "Quantity", "UnitPrice", "LineCategory",
            "IsNetProductRevenueEligible"
        ]
    ].head(20)
)

,row_count
Description,
str,540454
float,1454
int,1


Non-text descriptions: 1


,SourceRowID,InvoiceNo,StockCode,Description,Quantity,UnitPrice,LineCategory,IsNetProductRevenueEligible
420391,420393,572891,23343,20713,-400,0.0,Product,False


In [3]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import csv

project_root = Path(
    r"D:\project  whith Alex\E-Commerce Sales & Customer Analytics"
)

analysis_dir = project_root / "data" / "processed" / "analysis"
zip_path = project_root / "data" / "processed" / "online_retail_dashboard_data.zip"

manifest_path = analysis_dir / "export_manifest.csv"
assert manifest_path.is_file(), f"File not found: {manifest_path}"

with manifest_path.open(encoding="utf-8-sig", newline="") as file:
    required_files = [
        row["File"] for row in csv.DictReader(file)
    ]

required_files += [
    "export_manifest.csv",
    "analysis_decision_log.md"
]

missing_files = [
    name for name in required_files
    if not (analysis_dir / name).is_file()
]

assert not missing_files, f"Missing files: {missing_files}"

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for name in required_files:
        archive.write(analysis_dir / name, arcname=name)

with ZipFile(zip_path, "r") as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)

print(f"ZIP ready: {zip_path}")
print(f"Files included: {len(required_files)}")
print(f"Size: {zip_path.stat().st_size / (1024 ** 2):.2f} MB")

ZIP ready: D:\project  whith Alex\E-Commerce Sales & Customer Analytics\data\processed\online_retail_dashboard_data.zip
Files included: 23
Size: 0.83 MB
